# Aula 05: Visualização de Dados e Storytelling Executivo
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

Na última aula da capacitação, vamos transformar nossas análises em gráficos prontos para apresentação usando `matplotlib` e `seaborn`:
1. **Gráfico de Linha:** Evolução mensal do Faturamento ao longo de 2035 (identificando o pico de fim de ano!).
2. **Gráfico de Barras Horizontais:** Ranking de Faturamento e Lucro por Vendedor.
3. **Boxplot:** Distribuição do Ticket de Venda por Setor.
4. **Heatmap (Mapa de Calor):** Faturamento por Estado e Tipo de Cliente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import get_close_matches

# Configurando o estilo visual dos gráficos
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)

# Carregando e limpando a base
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

for col in ['faturamento', 'margem_de_lucro', 'lucro']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[df['lucro'] > df['faturamento'], 'lucro'] = df['faturamento'] * df['margem_de_lucro']
df.loc[(df['margem_de_lucro'] == 0) & (df['lucro'] > 0), 'margem_de_lucro'] = df['lucro'] / df['faturamento']
df.loc[(df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01, 'faturamento'] = df['lucro'] / df['margem_de_lucro']

df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro']).round(2)
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro']).round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento']).round(2)
df = df.drop_duplicates().reset_index(drop=True)
df['mes'] = df['data_da_venda'].dt.month

## 1. Sazonalidade: Evolução Mensal do Faturamento (Gráfico de Linha)
Como as vendas da Campe Supply se comportam ao longo dos 12 meses de 2035?

In [ ]:
vendas_mensais = df.groupby('mes')[['faturamento', 'lucro']].sum().reset_index()

plt.figure(figsize=(11, 5))
sns.lineplot(data=vendas_mensais, x='mes', y='faturamento', marker='o', linewidth=2.5, color='#1f77b4')
plt.title('Evolução Mensal do Faturamento — Campe Supply (2035)', fontsize=14, fontweight='bold')
plt.xlabel('Mês do Ano', fontsize=12)
plt.ylabel('Faturamento Total (R$)', fontsize=12)
plt.xticks(range(1, 13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
plt.tight_layout()
plt.show()

**Insight de Negócio:** Repare no salto expressivo nos meses de **Novembro e Dezembro**, onde os tickets médios sobem para a faixa de R$ 2.000+ por venda!

## 2. Ranking de Faturamento por Vendedor (Gráfico de Barras)

In [ ]:
ranking_vend = df.groupby('vendedor')['faturamento'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=ranking_vend, x='faturamento', y='vendedor', palette='viridis')
plt.title('Ranking de Faturamento Total por Vendedor', fontsize=14, fontweight='bold')
plt.xlabel('Faturamento Acumulado (R$)', fontsize=12)
plt.ylabel('Vendedor', fontsize=12)

# Adicionando os rótulos de valor na ponta de cada barra
for container in ax.containers:
    ax.bar_label(container, fmt=' R$ %.0f', padding=3)

plt.xlim(0, ranking_vend['faturamento'].max() * 1.15)
plt.tight_layout()
plt.show()

## 3. Distribuição do Valor das Vendas por Setor (Boxplot)
O Boxplot é excelente para mostrar mediana, quartis e as vendas de alto valor de fim de ano em cada setor:

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='setor', y='faturamento', palette='Set2')
plt.title('Distribuição do Faturamento por Setor', fontsize=14, fontweight='bold')
plt.xlabel('Setor', fontsize=12)
plt.ylabel('Valor da Venda (R$)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Mapa de Calor (Heatmap): Estado vs. Canal de Cliente
Quais estados concentram mais faturamento no Atacado, Atacarejo e Varejo?

In [ ]:
matriz_estado_cliente = pd.pivot_table(
    df, values='faturamento', index='estado', columns='cliente', aggfunc='sum'
)

plt.figure(figsize=(9, 6))
sns.heatmap(matriz_estado_cliente, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Faturamento (R$)'})
plt.title('Mapa de Calor: Faturamento por Estado e Tipo de Cliente', fontsize=14, fontweight='bold')
plt.xlabel('Canal do Cliente', fontsize=12)
plt.ylabel('Estado', fontsize=12)
plt.tight_layout()
plt.show()